# ML Briefs 3 demo

In [1]:
import numpy as np
import plotly.express as px
import pandas as pd
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from utils import read_data

# Introduction

In [2]:
# configuration parameters
subject = 4
movement = "AEBPSA"
sensor = 0

In [3]:
# load the data
#ts, metadata = read_data(subject=subject, movement=movement)
import json
ts = np.load("../Data_Smartarm_MC/smartarm_gapfree/smartarm_04_ABR.npy")
filename_meta = "../Data_Smartarm_MC/MetaData/smartarm_04_ABR.json"
with open(filename_meta, 'r') as f:
    metadata = json.load(f)
ts_sensor = ts[sensor, :, :]
print(ts_sensor.shape)
print(metadata)

(4001, 3)
{'Data_filename': 'smartarm_04_ABR.npy', 'Patient_info': {'ID': 4, 'Gender': 'Male', 'Age': 30, 'Size': 175.0, 'Weight': 77.0, 'Laterality': 'Left-Handed', 'BMI': 25.1428571428571}, 'Movement_info': {'movement_ID': 'ABR', 'Position': None, 'Movement': {'Limb': 'Arm', 'Laterality': None, 'Action': 'Rotation'}, 'Plan': None, 'Hand': {'Laterality': None, 'Position': None}}, 'Movement_label': {'Iteration_1': [313.0, 1186.0], 'Iteration_2': [1303.0, 3076.0], 'Iteration_3': [3255.0, 3080.0]}}


In [4]:
print(ts.shape)

(34, 4001, 3)


In [5]:
ts_sensor_x, ts_sensor_y, ts_sensor_z = ts_sensor[:, 0], ts_sensor[:, 1], ts_sensor[:, 2]
time = np.arange(len(ts_sensor))

N_desired = 100  # number of desired frames
h = len(ts_sensor_x) // N_desired
print(f"{h = }")

ts_sensor_xx = np.array(list(ts_sensor_x[::h]) + [list(ts_sensor_x)[-1]])
ts_sensor_yy = np.array(list(ts_sensor_y[::h]) + [list(ts_sensor_y)[-1]])
ts_sensor_zz = np.array(list(ts_sensor_z[::h]) + [list(ts_sensor_z)[-1]])
tt = np.array(list(time[::h]) + [list(time)[-1]])
N = len(ts_sensor_xx) # actual number of frames obtained
print(f"{N = }")

h = 40
N = 102


# [ML Briefs 3] Subplots (without animation)

https://plotly.com/python/subplots/

In [6]:
fig = make_subplots(
    rows=4,
    cols=1,
    row_heights=[0.7, 0.1, 0.1, 0.1],
    specs=[
        [{"type": "scene"}],
        [{"type": "xy"}],
        [{"type": "xy"}],
        [{"type": "xy"}]
    ],
    subplot_titles=('Trajectory', 'Time series X', 'Time series Y', 'Time series Z')
)
fig.add_trace(
    go.Scatter3d(
        x=ts_sensor_x,
        y=ts_sensor_y,
        z=ts_sensor_z,
        name="trajectory",
        mode="lines+markers",
        marker={'size':2}
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=time,
        y=ts_sensor_x,
        name="x",
    ),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=time,
        y=ts_sensor_y,
        name="y",
    ),
    row=3, col=1
)
fig.add_trace(
    go.Scatter(
        x=time,
        y=ts_sensor_z,
        name="z",
    ),
    row=4, col=1
)
fig.update_yaxes(title_text='x', row=2, col=1)
fig.update_yaxes(title_text='y', row=3, col=1)
fig.update_yaxes(title_text='z', row=4, col=1)
fig.update_xaxes(title_text='time stamp', row=4, col=1)
fig.update_layout(height=1000, width=800, title_text="IPOL demo")
fig.show()

In [7]:
fig = make_subplots(
    row_heights=[0.7, 0.1, 0.1, 0.1],
    column_widths=[0.5, 0.5],
    rows=4, cols=2,
    specs=[[{"rowspan": 4, "type": "scene"}, {"type": "scene"}],
           [None, {"type": "xy"}],
           [None, {"type": "xy"}],
           [None, {"type": "xy"}],
           ],
    print_grid=True)
fig.add_trace(go.Scatter3d(x=[1, 2], y=[1, 2], z=[1, 1], name="(1,1)"), row=1, col=1)
fig.add_trace(go.Scatter3d(x=[1, 2], y=[1, 2], z=[1, 1], name="(1,1)"), row=1, col=2)
fig.add_trace(go.Scatter(x=[1, 2], y=[1, 2], name="(1,1)"), row=2, col=2)
fig.add_trace(go.Scatter(x=[1, 2], y=[1, 2], name="(1,1)"), row=3, col=2)
fig.add_trace(go.Scatter(x=[1, 2], y=[1, 2], name="(1,1)"), row=4, col=2)
fig.update_layout(height=800, width=1000, title_text="specs examples")
fig.show()

This is the format of your plot grid:
⎡ (1,1) scene  ⎤  [ (1,2) scene2 ]
⎢      :       ⎟  [ (2,2) x,y    ]
⎢      :       ⎟  [ (3,2) x2,y2  ]
⎣      :       ⎦  [ (4,2) x3,y3  ]



# Animation

## Plotly example: animation on 2D curvature

Source: [plotly](https://plotly.com/python/animations/) - _Moving Point on a Curve_

In [8]:
# Generate curve data
t = np.linspace(-1, 1, 100)
x = t + t ** 2
y = t - t ** 2
xm = np.min(x) - 1.5
xM = np.max(x) + 1.5
ym = np.min(y) - 1.5
yM = np.max(y) + 1.5
N = 50
s = np.linspace(-1, 1, N)
xx = s + s ** 2
yy = s - s ** 2

# Create figure
fig = go.Figure(
    data=[go.Scatter(x=x, y=y,
                     mode="lines",
                     line=dict(width=2, color="blue")),
          go.Scatter(x=x, y=y,
                     mode="lines",
                     line=dict(width=2, color="blue"))],
    layout=go.Layout(
        xaxis=dict(range=[xm, xM], autorange=False, zeroline=False),
        yaxis=dict(range=[ym, yM], autorange=False, zeroline=False),
        title_text="Kinematic Generation of a Planar Curve", hovermode="closest",
        updatemenus=[dict(type="buttons",
                          buttons=[dict(label="Play",
                                        method="animate",
                                        args=[None])])]),
    frames=[go.Frame(
        data=[go.Scatter(
            x=[xx[k]],
            y=[yy[k]],
            mode="markers",
            marker=dict(color="red", size=10))])

        for k in range(N)]
)

fig.show()

## [ML Briefs 3] Animation on 3D scatter

In [9]:
# Generate curve data

xm = np.min(ts_sensor_x) - 1.5
xM = np.max(ts_sensor_x) + 1.5
ym = np.min(ts_sensor_y) - 1.5
yM = np.max(ts_sensor_y) + 1.5
zm = np.min(ts_sensor_z) - 1.5
zM = np.max(ts_sensor_z) + 1.5

# Create figure
fig = go.Figure(
    data=[go.Scatter3d(x=ts_sensor_x, y=ts_sensor_y, z=ts_sensor_z,
                     mode="lines",
                     line=dict(width=2, color="blue")),
          go.Scatter3d(x=ts_sensor_x, y=ts_sensor_y, z=ts_sensor_z,
                     mode="lines",
                     line=dict(width=2, color="blue"))
                     ],
    layout=go.Layout(
        xaxis=dict(range=[xm, xM], autorange=False, zeroline=False),
        yaxis=dict(range=[ym, yM], autorange=False, zeroline=False),
        #zaxis=dict(range=[zm, zM], autorange=False, zeroline=False),
        title_text="3D animation", hovermode="closest",
        updatemenus=
            [dict(type="buttons",
                          buttons=[dict(label="Play",
                                        method="animate",
                                        args=[None]),
                                     dict(
                    label="Pause",
                    method="animate",
                    args=[
                        None,
                        {
                            "frame": {"duration": 0, "redraw": False},
                            "mode": "immediate"
                        }
                    ]
                )   
                                        ]
                ),
            ]
            ),
    frames=[go.Frame(
        data=[go.Scatter3d(
            x=[ts_sensor_xx[k]],
            y=[ts_sensor_yy[k]],
            z=[ts_sensor_zz[k]],
            mode="markers",
            marker=dict(color="red", size=4))],
    )
        for k in range(N)
        ]
)
#fig.write_html("output.html", include_plotlyjs='cdn')
fig.show()


## [Plotly] Best example of animation

Source: [plotly](https://plotly.com/python/animations/) - _Using a Slider and Buttons_

In [10]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv"
dataset = pd.read_csv(url)

years = ["1952", "1962", "1967", "1972", "1977", "1982", "1987", "1992", "1997", "2002",
         "2007"]

# make list of continents
continents = []
for continent in dataset["continent"]:
    if continent not in continents:
        continents.append(continent)

# make figure
fig_dict = {
    "data": [],
    "layout": {},
    "frames": []
}

# fill in most of layout
fig_dict["layout"]["xaxis"] = {"range": [30, 85], "title": "Life Expectancy"}
fig_dict["layout"]["yaxis"] = {"title": "GDP per Capita", "type": "log"}
fig_dict["layout"]["hovermode"] = "closest"
fig_dict["layout"]["updatemenus"] = [
    {
        "buttons": [
            {
                "args": [None, {"frame": {"duration": 500, "redraw": False},
                                "fromcurrent": True, "transition": {"duration": 300,
                                                                    "easing": "quadratic-in-out"}}],
                "label": "Play",
                "method": "animate"
            },
            {
                "args": [[None], {"frame": {"duration": 0, "redraw": False},
                                  "mode": "immediate",
                                  "transition": {"duration": 0}}],
                "label": "Pause",
                "method": "animate"
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }
]

sliders_dict = {
    "active": 0,
    "yanchor": "top",
    "xanchor": "left",
    "currentvalue": {
        "font": {"size": 20},
        "prefix": "Year:",
        "visible": True,
        "xanchor": "right"
    },
    "transition": {"duration": 300, "easing": "cubic-in-out"},
    "pad": {"b": 10, "t": 50},
    "len": 0.9,
    "x": 0.1,
    "y": 0,
    "steps": []
}

# make data
year = 1952
for continent in continents:
    dataset_by_year = dataset[dataset["year"] == year]
    dataset_by_year_and_cont = dataset_by_year[
        dataset_by_year["continent"] == continent]

    data_dict = {
        "x": list(dataset_by_year_and_cont["lifeExp"]),
        "y": list(dataset_by_year_and_cont["gdpPercap"]),
        "mode": "markers",
        "text": list(dataset_by_year_and_cont["country"]),
        "marker": {
            "sizemode": "area",
            "sizeref": 200000,
            "size": list(dataset_by_year_and_cont["pop"])
        },
        "name": continent
    }
    fig_dict["data"].append(data_dict)

# make frames
for year in years:
    frame = {"data": [], "name": str(year)}
    for continent in continents:
        dataset_by_year = dataset[dataset["year"] == int(year)]
        dataset_by_year_and_cont = dataset_by_year[
            dataset_by_year["continent"] == continent]

        data_dict = {
            "x": list(dataset_by_year_and_cont["lifeExp"]),
            "y": list(dataset_by_year_and_cont["gdpPercap"]),
            "mode": "markers",
            "text": list(dataset_by_year_and_cont["country"]),
            "marker": {
                "sizemode": "area",
                "sizeref": 200000,
                "size": list(dataset_by_year_and_cont["pop"])
            },
            "name": continent
        }
        frame["data"].append(data_dict)

    fig_dict["frames"].append(frame)
    slider_step = {"args": [
        [year],
        {"frame": {"duration": 300, "redraw": False},
         "mode": "immediate",
         "transition": {"duration": 300}}
    ],
        "label": year,
        "method": "animate"}
    sliders_dict["steps"].append(slider_step)


fig_dict["layout"]["sliders"] = [sliders_dict]

fig = go.Figure(fig_dict)

#fig.write_html("output.html", include_plotlyjs='cdn')

fig.show()

# Animation on subplots

## Toy example

https://chart-studio.plotly.com/~empet/15243/animating-traces-in-subplotsbr/#/

In [11]:
fig = make_subplots(rows=1, cols=2, subplot_titles = ('Subplot (1,1)', 'Subplot(1,2)'))

fig.add_trace(go.Scatter(
          x= [-2.0, -1.0, 0.01, 1.0, 2.0, 3.0],
          y= [4, 1, 1, 1, 4, 9],
          mode = 'markers+lines',
          hoverinfo='name',
          legendgroup= 'f1',
          line_color= 'rgb(255, 79, 38)',
          name= 'f1',
          showlegend= True), row=1, col=1);


fig.add_trace(go.Scatter(
          x= [-2.0, -1.0, 0.01, 1.0, 2.0, 3.0],
          y= [3, 8.3, 5.5, 4.24, 6.7, 7],
          mode = 'markers+lines',
          hoverinfo='name',
          legendgroup= 'f1p',
          line_color= 'rgb(79, 38, 255)',
          name= 'f1p',
          showlegend= True), row=1, col=1);

fig.add_trace(go.Scatter(
            x = [-2.0, -1.0, 0.01, 1.0, 2.0, 3.0],
            y = [2.5, 3, 6, 1, 2.5, 8.31],
            name= 'f2',
            showlegend= True,
            hoverinfo= 'name',
            legendgroup= 'f2',
            mode = 'markers+lines',
            line_color='green',
            ), row=1, col=2);

fig.update_layout(width=700, height=475)
fig.update_yaxes(range=[0, 9]);#this line updates both yaxis, and yaxis2 range

number_frames = 10
frames = [dict(
               name = k,
               data = [go.Scatter(y= 4+5*np.random.rand(6)),#update the trace 1 in (1,1)
                       go.Scatter(y = 2+6*np.random.rand(6)), #update the second trace in (1,1)
                       go.Scatter(y= 1+7*np.random.rand(10))#update the trace in (1,2)
                       ],
               traces=[0, 1, 2] # the elements of the list [0,1,2] give info on the traces in fig.data
                                      # that are updated by the above three go.Scatter instances
              ) for k in range(number_frames)]

updatemenus = [dict(type='buttons',
                    buttons=[dict(label='Play',
                                  method='animate',
                                  args=[[f'{k}' for k in range(10)], 
                                         dict(frame=dict(duration=500, redraw=False), 
                                              transition=dict(duration=0),
                                              easing='linear',
                                              fromcurrent=True,
                                              mode='immediate'
                                                                 )])],
                    direction= 'left', 
                    pad=dict(r= 10, t=85), 
                    showactive =True, x= 0.1, y= 0, xanchor= 'right', yanchor= 'top')
            ]

sliders = [{'yanchor': 'top',
            'xanchor': 'left', 
            'currentvalue': {'font': {'size': 16}, 'prefix': 'Frame: ', 'visible': True, 'xanchor': 'right'},
            'transition': {'duration': 500.0, 'easing': 'linear'},
            'pad': {'b': 10, 't': 50}, 
            'len': 0.9, 'x': 0.1, 'y': 0, 
            'steps': [{'args': [[k], {'frame': {'duration': 500.0, 'easing': 'linear', 'redraw': False},
                                      'transition': {'duration': 0, 'easing': 'linear'}}], 
                       'label': k, 'method': 'animate'} for k in range(10)       
                    ]}]

fig.update(frames=frames),
fig.update_layout(updatemenus=updatemenus,
                  sliders=sliders);
fig.show() #in jupyter notebook


## ML Briefs 3

In [12]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Subplot (1,1)', 'Subplot(2,1)')
)

fig.add_trace(
    go.Scatter(
        x=time,
        y=ts_sensor_x,
        mode='markers+lines',
        hoverinfo='name',
        legendgroup='f1',
        line_color='rgb(255, 79, 38)',
        name='f1',
        showlegend=True
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=time,
        y=ts_sensor_y,
        mode='markers+lines',
        hoverinfo='name',
        legendgroup='f1',
        line_color='blue',
        name='f1',
        showlegend=True
    ),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(
        x=[tt[0]],
        y=[ts_sensor_yy[0]],
        mode="markers",
        marker=dict(color="red", size=4)),
    row=1, col=2
)

frames = [
    dict(
        data=[
            go.Scatter(visible=True),
            go.Scatter(visible=True),
            go.Scatter(
                x=[tt[k]],
                y=[ts_sensor_yy[k]],
                mode="markers",
                marker=dict(color="red", size=4))
        ],
        traces=[0, 1, 2]
    )
    for k in range(N)
]

updatemenus = [
    dict(
        type='buttons',
        buttons=[
            dict(
                label="Play",
                method="animate",
                args=[None]
            ),
            dict(
                label="Pause",
                method="animate",
                args=[
                    None,
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate"
                    }
                ]
            )   
        ],
        direction='left',
        pad=dict(r=10, t=85), 
        showactive=True,
        x=0.1,
        y=0,
        xanchor='right',
        yanchor='top'
    )
]

sliders = [
    {
        'yanchor': 'top',
        'xanchor': 'left',
        'currentvalue': {
            'font': {'size': 16},
            'prefix': 'Frame: ',
            'visible': True,
            'xanchor': 'right'
        },
        'transition': {'duration': 500, 'easing': 'linear'},
        'pad': {'b': 10, 't': 50}, 
        'len': 0.9,
        'x': 0.1,
        'y': 0, 
        'steps': [
            {'args': [[k],
                {'frame': {'duration': 500, 'easing': 'linear', 'redraw': False},
                                      'transition': {'duration': 0, 'easing': 'linear'
            }}],
            'label': k, 'method': 'animate'} for k in range(N)       
        ]
    }
]

fig.update(
    frames=frames
)
fig.update_layout(
    updatemenus=updatemenus,
    sliders=sliders
)
fig.show()